## DEP Lab Experiment 5

In [20]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_breast_cancer # Breast cancer dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, root_mean_squared_error, accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, chi2, f_classif, RFE, SequentialFeatureSelector

print(f"Dataset shape: {X.shape}")

Dataset shape: (569, 30)


### Part 1: Feature Selection on Structured Data

#### Step 1: Baseline Model

Training classifier (Logistic Regression)

In [21]:
cancer_data = load_breast_cancer()
X = pd.DataFrame(cancer_data.data, columns=cancer_data.feature_names)
y = cancer_data.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize the data (crucial for Logistic Regression, LASSO, and ANOVA)
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X.columns)

# Helper function to train and evaluate our models quickly
def evaluate_model(X_train_subset, X_test_subset, y_train, y_test, method_name):
    model = LogisticRegression(random_state=42, max_iter=1000)
    model.fit(X_train_subset, y_train)
    preds = model.predict(X_test_subset)
    acc = accuracy_score(y_test, preds)
    print(f"{method_name} Accuracy: {acc:.4f}")
    return acc

Recording accuracy

In [22]:
baseline_acc = evaluate_model(X_train_scaled, X_test_scaled, y_train, y_test, "")

 Accuracy: 0.9737


#### Filter Methods

Correlation-based selection, Chi-Square Test & ANOVA test 

In [23]:
#Correlation-based selection (Pearson)
corr_df = X_train.copy()
corr_df['Target'] = y_train
top_10_corr = corr_df.corr()['Target'].abs().drop('Target').nlargest(10).index

X_train_corr = X_train_scaled[top_10_corr]
X_test_corr = X_test_scaled[top_10_corr]
evaluate_model(X_train_corr, X_test_corr, y_train, y_test, "Correlation-based")

#Chi-square test (Must use unscaled X_train because it doesn't accept negative values)
chi2_selector = SelectKBest(score_func=chi2, k=10)
X_train_chi2 = chi2_selector.fit_transform(X_train, y_train)
X_test_chi2 = chi2_selector.transform(X_test)

X_train_chi2_scaled = scaler.fit_transform(X_train_chi2)
X_test_chi2_scaled = scaler.transform(X_test_chi2)
evaluate_model(X_train_chi2_scaled, X_test_chi2_scaled, y_train, y_test, "Chi-Square Test")

#ANOVA F-test
anova_selector = SelectKBest(score_func=f_classif, k=10)
X_train_anova = anova_selector.fit_transform(X_train_scaled, y_train)
X_test_anova = anova_selector.transform(X_test_scaled)
evaluate_model(X_train_anova, X_test_anova, y_train, y_test, "ANOVA F-test")

Correlation-based Accuracy: 0.9737
Chi-Square Test Accuracy: 0.9649
ANOVA F-test Accuracy: 0.9737


0.9736842105263158

#### Step 3: Wrapper Methods

Recursive Feature Elimination (RFE) & Sequential Forward Selection

In [24]:
base_estimator = LogisticRegression(random_state=42, max_iter=1000)

# Recursive Feature Elimination (RFE)
# Starts with all features and removes the weakest ones step-by-step
rfe = RFE(estimator=base_estimator, n_features_to_select=10)
X_train_rfe = rfe.fit_transform(X_train_scaled, y_train)
X_test_rfe = rfe.transform(X_test_scaled)
evaluate_model(X_train_rfe, X_test_rfe, y_train, y_test, "RFE")

# Sequential Forward Selection (SFS)
# Starts with 0 features and adds the most impactful ones step-by-step
sfs = SequentialFeatureSelector(estimator=base_estimator, n_features_to_select=10, direction='forward')
X_train_sfs = sfs.fit_transform(X_train_scaled, y_train)
X_test_sfs = sfs.transform(X_test_scaled)
evaluate_model(X_train_sfs, X_test_sfs, y_train, y_test, "Sequential Forward Selection")

RFE Accuracy: 0.9737
Sequential Forward Selection Accuracy: 0.9737


0.9736842105263158

#### Step-4: Embedded Methods

L1 (Lasso Regularization) & Random Forest 

In [25]:
# LASSO (L1 Regularization)
lasso_model = LogisticRegression(penalty='l1', solver='liblinear', random_state=42, max_iter=1000)
lasso_model.fit(X_train_scaled, y_train)
lasso_preds = lasso_model.predict(X_test_scaled)
lasso_acc = accuracy_score(y_test, lasso_preds)

# Count how many features LASSO actually kept (coefficients != 0)
features_kept = np.sum(lasso_model.coef_ != 0)
print(f"LASSO (L1) Accuracy: {lasso_acc:.4f} (Kept {features_kept} features)")

# 2. Tree-based feature importance (Random Forest)
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train_scaled, y_train)
rf_preds = rf_model.predict(X_test_scaled)
rf_acc = accuracy_score(y_test, rf_preds)
print(f"Random Forest Accuracy: {rf_acc:.4f}")

# Display top 10 features identified by Random Forest
importances = pd.Series(rf_model.feature_importances_, index=X.columns)
print("\nTop 5 Features according to Random Forest:")
print(importances.nlargest(5))

LASSO (L1) Accuracy: 0.9737 (Kept 14 features)


e:\COLLEGE\Semester-6\DEP\Assignments\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
e:\COLLEGE\Semester-6\DEP\Assignments\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


Random Forest Accuracy: 0.9649

Top 5 Features according to Random Forest:
worst area              0.153892
worst concave points    0.144663
mean concave points     0.106210
worst radius            0.077987
mean concavity          0.068001
dtype: float64


### Part-2: Text Feature Extraction